In [27]:
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# 1. Setup your connection details
user = 'root'
password = 'Niknduku@17'  
host = 'localhost'
database = 'candy_distributor'

# 2. Build the URL correctly
url_object = URL.create(
    "mysql+mysqlconnector",
    username=user,
    password=password,
    host=host,
    database=database
)

# 3. Create the connection engine
engine = create_engine(url_object)

# 4. Pull the data from your SQL View
query = "SELECT * FROM Sales_Analysis_View"
df = pd.read_sql(query, engine)

# 5. Check if it worked
print("Connection Successful! Here is the data:")
print(df.head())

Connection Successful! Here is the data:
                       Order_ID  Order_Date   Ship_Date Postal_Code  \
0  US-2021-103800-CHO-MIL-31000  2021-01-03  2026-06-30       77095   
1  US-2021-112326-CHO-TRI-54000  2021-01-04  2026-07-01       60540   
2  US-2021-112326-CHO-NUT-13000  2021-01-04  2026-07-01       60540   
3  US-2021-112326-CHO-SCR-58000  2021-01-04  2026-07-01       60540   
4  US-2021-141817-CHO-TRI-54000  2021-01-05  2026-07-05       19143   

    Division                       Product_Name  Sales_Amount  Units  \
0  Chocolate         Wonka Bar - Milk Chocolate          6.50      2   
1  Chocolate  Wonka Bar - Triple Dazzle Caramel          7.50      2   
2  Chocolate  Wonka Bar - Nutty Crunch Surprise         10.47      3   
3  Chocolate     Wonka Bar -Scrumdiddlyumptious         10.80      3   
4  Chocolate  Wonka Bar - Triple Dazzle Caramel         11.25      3   

   Gross_Profit  Cost          Factory  Unit_Price  Unit_Cost  Target_2024  
0          4.22  2.28 

In [28]:
# Just re-run this cell
query = "SELECT * FROM Sales_Analysis_View"
df = pd.read_sql(query, engine)

# Check if Postal_Code is now there
print(df.columns)

Index(['Order_ID', 'Order_Date', 'Ship_Date', 'Postal_Code', 'Division',
       'Product_Name', 'Sales_Amount', 'Units', 'Gross_Profit', 'Cost',
       'Factory', 'Unit_Price', 'Unit_Cost', 'Target_2024'],
      dtype='object')


1. Shipping Efficiency
Find out how many days it took to ship the candy.

In [29]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Ship_Date'] = pd.to_datetime(df['Ship_Date'])

df['Days_to_Ship'] = (df['Ship_Date'] - df['Order_Date']).dt.days

print(df[['Order_Date', 'Ship_Date', 'Days_to_Ship']].head())


  Order_Date  Ship_Date  Days_to_Ship
0 2021-01-03 2026-06-30          2004
1 2021-01-04 2026-07-01          2004
2 2021-01-04 2026-07-01          2004
3 2021-01-04 2026-07-01          2004
4 2021-01-05 2026-07-05          2007


2. Profit Margin %

In [30]:
df['Profit_Margin_pct'] = (df['Gross_Profit']/ df['Sales_Amount']) *100
df['Profit_Margin_pct'].head()

0    64.923077
1    65.333333
2    71.346705
3    69.444444
4    65.333333
Name: Profit_Margin_pct, dtype: float64

3. Target Achievement
How close are we to the goal?

In [31]:
division_performance = df.groupby('Division').agg({
    'Sales_Amount' : 'sum',
    'Target_2024' : 'mean'
}).reset_index()

#Calculates the percentage
division_performance['pct_of_target'] = ((division_performance['Sales_Amount']/ division_performance['Target_2024'])*100).round(2)

#Sort them so the best performers are at the top
division_performance= division_performance.sort_values(by='pct_of_target', ascending=False)

print(division_performance[['Division', 'Sales_Amount', 'Target_2024', 'pct_of_target']])

    Division  Sales_Amount  Target_2024  pct_of_target
0  Chocolate     131692.90      27000.0         487.75
1      Other       9663.25       3000.0         322.11
2      Sugar        427.48      15000.0           2.85


4.Find Orders where profit is negative

In [32]:
loss_orders = df[df['Gross_Profit'] < 0]
print(f"Number of loss-making orders: {len(loss_orders)}")

Number of loss-making orders: 0


In [33]:
#Extract the years
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Order_Year'] = df['Order_Date'].dt.year
df['Order_Month'] = df['Order_Date'].dt.month_name()


In [34]:
# We will flag any item with a margin over 50% as 'High Margin'
df['Profit_Margin_pct'] = (df['Gross_Profit']/ df['Sales_Amount']) *100

df['Profit_Category'] = df['Profit_Margin_pct'].apply(lambda  x: 'High Margin' if x >= 50 else 'Standard Margin')


In [35]:
df['Cost_Check'] = df['Units'] * df['Unit_Cost']

print(df[['Order_Year','Order_Month', 'Profit_Category', 'Cost_Check']].head())

   Order_Year Order_Month Profit_Category  Cost_Check
0        2021     January     High Margin        2.28
1        2021     January     High Margin        2.60
2        2021     January     High Margin        3.00
3        2021     January     High Margin        3.30
4        2021     January     High Margin        3.90


In [36]:
print(df.columns)

Index(['Order_ID', 'Order_Date', 'Ship_Date', 'Postal_Code', 'Division',
       'Product_Name', 'Sales_Amount', 'Units', 'Gross_Profit', 'Cost',
       'Factory', 'Unit_Price', 'Unit_Cost', 'Target_2024', 'Days_to_Ship',
       'Profit_Margin_pct', 'Order_Year', 'Order_Month', 'Profit_Category',
       'Cost_Check'],
      dtype='object')


In [37]:
df.to_csv('Candy_Sales_Analytical.csv', index=False)